In [0]:
import pyspark.sql.functions as F

# Carga de datos

In [0]:
beneficiarios = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/Beneficiarios/")

In [0]:
establecimientos = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/establecimientos/")

In [0]:
# Educacion (estadisticas) por municipio https://www.datos.gov.co/Educaci-n/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR-B-SICA/nudc-7mev/about_data
mpio = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/mpio/")

In [0]:
# Pruebas saber 11 https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe/about_data
Pruebas = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/saber/")

In [0]:
trabajo = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/trabajo/")

In [0]:
victima_matr = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/victima_matr/")

# Limpieza para join

In [0]:
# Columnas con mismo nombre
establecimientos = establecimientos.withColumnRenamed("AÑO","AÑO_establecimientos")
mpio = mpio.withColumnRenamed("AÑO","AÑO_mpio")

trabajo = trabajo.withColumnRenamed("REGION","REGION_trabajo")
victima_matr = victima_matr.withColumnRenamed("REGION","REGION_victima_matr")

trabajo = trabajo.drop("MUNICIPIO","DEPARTAMENTO")
victima_matr = victima_matr.drop("MUNICIPIO","DEPARTAMENTO")

establecimientos = establecimientos.drop("COD_DANE_DEPARTAMENTO","COD_SECRETARIA","SECRETARIA","CODIGO_DANE","CANTIDAD_SEDES")

Pruebas = Pruebas.drop("PERIODO","COLE_COD_DANE_ESTABLECIMIENTO","COLE_COD_DEPTO_UBICACION","COLE_DEPTO_UBICACION","ESTU_COD_RESIDE_DEPTO","ESTU_DEPTO_RESIDE","AVG_DEPTO")

beneficiarios= beneficiarios.drop("id_familia","codigo_dep_atencion","nombre_dep_atencion","municipio_atencion_key","etnia","etnia_reportada_flag")

## Agrupaciones

### Víctimas

In [0]:
victima_matr2 = victima_matr
victima_matr2 = victima_matr2.groupBy("COD_MUNICIPIO").agg(
    F.sum("FEMENINO").alias("FEMENINO"),
    F.sum("MASCULINO").alias("MASCULINO"),
    F.sum("total").alias("total")
)

### Trabajo

In [0]:
trabajo2 = (
    trabajo
    .groupBy(


        "ZONA",
        "ACTIVIDAD_PRINCIPAL_SEMANA",
        "OFICIOS_HOGAR",
        "RAZON_OFICIOS",
        "RAZON_TRABAJO",
        "CARGO_TRABAJO",
        "MESES_TRABAJADOS",
        "UBICACION_TRABAJO",
        "REALIZA_ALGUN_OFICIO_HOGAR",
        "TOTAL_ACT_NO_REMUNERADAS",
        "REALIZA_ACTIVIDAD_NO_REMUNERADA",
        



        "PARTICIPA_MERCADO_LABORAL",
        "ESTADO_LABORAL_INFANTIL",
        "COD_MUNICIPIO"

    )
    .agg(
        F.count("*").alias("TOTAL_PERSONAS"),
        F.sum("TOTAL_HORAS_OFICIO_HOGAR").alias("TOTAL_HORAS_OFICIO_HOGAR"),
        F.sum("TOTAL_HORAS_NO_REMUNERADAS").alias("TOTAL_HORAS_NO_REMUNERADAS"),
        F.sum("CARGA_TOTAL_TRABAJO").alias("CARGA_TOTAL_TRABAJO")
    )
)

### Pruebas

In [0]:
Pruebas2 = Pruebas.groupBy("COD_MUNICIPIO","ESTU_MCPIO_RESIDE","COLE_AREA_UBICACION","COLE_NATURALEZA","COLE_BILINGUE","COLE_JORNADA","COLE_GENERO","ESTU_PRIVADO_LIBERTAD").agg(
        
    F.max("PUNT_LECTURA_CRITICA").alias("MAX_PUNT_LECTURA_CRITICA"),
    F.min("PUNT_LECTURA_CRITICA").alias("MIN_PUNT_LECTURA_CRITICA"),
    F.avg("PUNT_LECTURA_CRITICA").alias("AVG_PUNT_LECTURA_CRITICA"),
    
    F.max("PUNT_C_NATURALES").alias("MAX_PUNT_C_NATURALES"),
    F.min("PUNT_C_NATURALES").alias("MIN_PUNT_C_NATURALES"),
    F.avg("PUNT_C_NATURALES").alias("AVG_PUNT_C_NATURALES"),
    
    F.max("PUNT_SOCIALES_CIUDADANAS").alias("MAX_PUNT_SOCIALES_CIUDADANAS"),
    F.min("PUNT_SOCIALES_CIUDADANAS").alias("MIN_PUNT_SOCIALES_CIUDADANAS"),
    F.avg("PUNT_SOCIALES_CIUDADANAS").alias("AVG_PUNT_SOCIALES_CIUDADANAS"),
    
    F.max("PUNT_MATEMATICAS").alias("MAX_PUNT_MATEMATICAS"),
    F.min("PUNT_MATEMATICAS").alias("MIN_PUNT_MATEMATICAS"),
    F.avg("PUNT_MATEMATICAS").alias("AVG_PUNT_MATEMATICAS"),
    
    F.max("PUNT_INGLES").alias("MAX_PUNT_INGLES"),
    F.min("PUNT_INGLES").alias("MIN_PUNT_INGLES"),
    F.avg("PUNT_INGLES").alias("AVG_PUNT_INGLES"),

    F.max("PUNT_GLOBAL").alias("MAX_PUNT_GLOBAL"),
    F.min("PUNT_GLOBAL").alias("MIN_PUNT_GLOBAL"),
    F.avg("PUNT_GLOBAL").alias("AVG_PUNT_GLOBAL")
    
)

### Mpio

In [0]:
mpio.createOrReplaceTempView("MPIOVIEW")
spark.sql("SELECT count(1) FROM MPIOVIEW ").show()

### Establecimientos

In [0]:
establecimientos2 = establecimientos.groupBy("COD_MUNICIPIO","AÑO_establecimientos","SECTOR","CARACTER","CALENDARIO","TIENE_WEB").agg(
    F.sum("TOTAL_MATRICULA").alias("TOTAL_MATRICULA")
)

### Beneficiarios

In [0]:
beneficiarios2 = beneficiarios.groupBy("COD_MUNICIPIO","nombre_mun_atencion","puntaje_sisben","estrato","rango_edad_ord","educacion_inicial_ord","acceso_educacion_ord","sin_trabajo_infantil_ord","seguridad_alimentaria_ord","alfabetizacion_ord","herramientas_digitales_ord","sin_hacinamiento_ord","sabe_leer_escribir_bin","estudia_actualmente_bin","ultimo_grado_cursado_ord","hogar_ingresos","hogar_ingresos_bin","hogar_ingresos_valor","hogar_ingresos_anomalia","hogar_falta_comida_bin").agg(
    F.count("*").alias("TOTAL_BENEFICIARIOS")
)

# Join

KEY "COD_MUNICIPIO"

In [0]:
Pruebas2.createOrReplaceTempView("DF1")
mpio.createOrReplaceTempView("DF2")
beneficiarios2.createOrReplaceTempView("DF3")
establecimientos2.createOrReplaceTempView("DF4")
victima_matr2.createOrReplaceTempView("DF5")
trabajo2.createOrReplaceTempView("DF6")
spark.sql("SELECT count(1) FROM DF1 ").show()
spark.sql("SELECT count(1) FROM DF2 ").show()
spark.sql("SELECT count(1) FROM DF3 ").show()
spark.sql("SELECT count(1) FROM DF4 ").show()
spark.sql("SELECT count(1) FROM DF5 ").show()
spark.sql("SELECT count(1) FROM DF6 ").show()

In [0]:
trabajo2.printSchema()

In [0]:
trabajo2.show()

In [0]:
from pyspark.sql.functions import broadcast

df01 = Pruebas2 \
    .join(broadcast(mpio), ["COD_MUNICIPIO"], "inner") \
    .join(broadcast(victima_matr2), ["COD_MUNICIPIO"], "left") \
    .join(
        beneficiarios2.groupBy("COD_MUNICIPIO")
        .sum("TOTAL_BENEFICIARIOS"),
        ["COD_MUNICIPIO"],
        "left"
    ) \
    .join(
        establecimientos2.groupBy("COD_MUNICIPIO",'SECTOR')
        .sum("TOTAL_MATRICULA"),
        ["COD_MUNICIPIO"],
        "left"
    ) \
    .join(
        trabajo2.groupBy("COD_MUNICIPIO").agg(
        F.sum("TOTAL_PERSONAS"),
        F.sum("CARGA_TOTAL_TRABAJO")),
        ["COD_MUNICIPIO"],
        "left"
    )

In [0]:
df01.write.mode("overwrite").parquet("/Volumes/workspace/pdge(saber-11)/saber11/GOLD/join1/")

In [0]:
#df01 = spark.read.parquet("/Volumes/workspace/pdge(saber-11)/saber11/GOLD/join1/")